# `groundinsight` — pandapower import walkthrough

End-to-end demonstration of the pandapower importer
`groundinsight.io.pandapower_import`:

1. Build a small pandapower MV network with one out-of-service branch and
   a low-voltage stub (so we can exercise the voltage-level filter).
2. Inspect the planned mapping with `preview_pandapower_import`.
3. Convert the network with `from_pandapower`.
4. Add a project-specific source and fault, solve.
5. Run an outage study on top of the imported network.

**Prerequisite.** The pandapower extra:

```
pip install "groundinsight[pandapower]"
```

In [ ]:
import matplotlib.pyplot as plt
import pandapower as pp
import polars as pl

import groundinsight as gi

print("groundinsight", gi.__version__)
print("pandapower  ", pp.__version__)

## 1. Build a small pandapower network

Five MV buses (20 kV) in a line, plus one LV stub bus (0.4 kV) connected
through a non-physical line so we can exercise the voltage-level filter.
One MV branch is set to `in_service=False` to demonstrate that the flag
propagates to `Branch.active = False` on the groundinsight side.

In [ ]:
net = pp.create_empty_network(name="ms_demo")

# 20 kV main feeder
b1 = pp.create_bus(net, vn_kv=20.0, name="bus_mv_1")
b2 = pp.create_bus(net, vn_kv=20.0, name="bus_mv_2")
b3 = pp.create_bus(net, vn_kv=20.0, name="bus_mv_3")
b4 = pp.create_bus(net, vn_kv=20.0, name="bus_mv_4")
b5 = pp.create_bus(net, vn_kv=20.0, name="bus_mv_5")

# 0.4 kV stub -- different voltage level, must be filtered out
b_lv = pp.create_bus(net, vn_kv=0.4, name="bus_lv_1")

# Cable parameters representative of a 20 kV XLPE run; the exact values
# do not matter for the demo because the importer uses the BranchType
# supplied via ImportDefaults for the earthing model.
common_kwargs = dict(r_ohm_per_km=0.25, x_ohm_per_km=0.6, c_nf_per_km=0.0, max_i_ka=0.4)

pp.create_line_from_parameters(net, from_bus=b1, to_bus=b2, length_km=1.5,
                               name="line_mv_12", **common_kwargs)
pp.create_line_from_parameters(net, from_bus=b2, to_bus=b3, length_km=2.0,
                               name="line_mv_23", **common_kwargs)
# Marked as out-of-service to demonstrate active-flag propagation.
pp.create_line_from_parameters(net, from_bus=b3, to_bus=b4, length_km=1.2,
                               name="line_mv_34_oos", in_service=False, **common_kwargs)
pp.create_line_from_parameters(net, from_bus=b4, to_bus=b5, length_km=1.8,
                               name="line_mv_45", **common_kwargs)
# Cross-voltage-level branch: pure filter-test stub.
pp.create_line_from_parameters(net, from_bus=b5, to_bus=b_lv, length_km=0.05,
                               name="line_mv_to_lv_stub", **common_kwargs)

print(f"{len(net.bus)} buses, {len(net.line)} lines")

## 2. Preview the import

`preview_pandapower_import` is a pure inspection: it does not build a
groundinsight `Network`. The returned DataFrame lists every pandapower bus and
line with `status="keep"` or `status="skip"` plus a `reason`, so the mapping
can be confirmed before committing to it.

In [ ]:
preview = gi.preview_pandapower_import(net, voltage_level_kV=20.0)
preview.sort("kind", "status", "pp_index").select(
    "kind", "status", "pp_index", "name", "length_km", "in_service", "reason"
)

**Expected.** Five 20 kV buses kept, the LV stub bus skipped with reason
`voltage_level_mismatch`, four MV–MV lines kept (one of them with
`in_service=False`, which becomes `active=False` after the import), and the
MV→LV stub line skipped with reason `endpoint_off_target_voltage_level`.

## 3. Convert with `from_pandapower`

Project-level defaults travel via `ImportDefaults`. The importer applies
`default_bus_type` to every imported bus and `default_branch_type` to
every imported line; soil resistivity `rho` and the frequency list are
the same for the whole import. The two types below are illustrative.

In [ ]:
bus_type = gi.BusType(
    name="ImportedBus",
    description="Default bus type for the demo import",
    system_type="Grounded",
    voltage_level=20.0,
    impedance_formula="rho * 0 + 1.0 + I * f * 0",
)

branch_type = gi.BranchType(
    name="ImportedCable",
    description="MV cable (R=0.25 ohm/km, omega*L=0.6 ohm/km at 50 Hz)",
    grounding_conductor=True,
    self_impedance_formula="(0.25 + I * 0.6) * l",
    mutual_impedance_formula="(0.0 + I * 0.6) * l",
)

defaults = gi.ImportDefaults(
    rho=100.0,
    frequencies=[50.0],
    default_bus_type=bus_type,
    default_branch_type=branch_type,
)

network = gi.from_pandapower(net, defaults=defaults, voltage_level_kV=20.0,
                             network_name="ms_demo_groundinsight")

print(f"imported {len(network.buses)} buses, {len(network.branches)} branches")
print()
print("active flags:")
for name, bus in network.buses.items():
    print(f"  bus    {name:<12} active={bus.active}")
for name, branch in network.branches.items():
    print(f"  branch {name:<22} active={branch.active}  length={branch.length:.2f} km")

## 4. Add a fault and solve

The importer never invents sources or fault locations — those are
project-specific. Here we inject 1 kA at `bus_mv_1` and place a
single-phase fault at `bus_mv_5`. The out-of-service `line_mv_34_oos` would
interrupt the feeder, so we re-activate it for this baseline solve; the outage
study below takes it out again as a scenario of its own.

In [ ]:
# Re-activate the out-of-service branch so the baseline solve has a closed path
network.branches["line_mv_34_oos"].active = True

gi.create_source(name="src", bus="bus_mv_1", values={50.0: 1000.0}, network=network)
gi.create_fault(name="F1", bus="bus_mv_5", network=network)

result = gi.run_fault(network, fault="F1")
network.res_buses(fault="F1").filter(pl.col("is_rms")).select(
    "bus_name", "EPR_V", "I_bus_A", "I_inj_A"
)

In [ ]:
fig = gi.plot_bus_voltages(result, title="EPR per bus, base case")
plt.show()

## 5. Outage study on the imported network

Take the imported network as it is and compare two what-if scenarios against
the base case in a single call. `OutageStudyResult.compare_buses` returns a
long-format DataFrame with absolute and relative deltas against the chosen
reference scenario (the base case by default).

The comparison frames join the scenarios on the frequency, and their
`frequency_Hz` column is text, with `"RMS"` marking the RMS rows — unlike
`res_buses()`, where `frequency_Hz` is a number and `is_rms` marks them.

In [ ]:
study = gi.run_outage_study(
    network,
    fault="F1",
    scenarios=[
        gi.Outage(
            name="line_mv_23_open",
            disabled_branches=["line_mv_23"],
            description="Cable section 2-3 cut out for maintenance",
        ),
        gi.Outage(
            name="bus3_isolated",
            disabled_buses=["bus_mv_3"],
            description="Substation 3 fully disconnected",
        ),
    ],
)

# RMS EPR per bus per scenario, with absolute and relative delta vs base
study.compare_buses().filter(pl.col("frequency_Hz") == "RMS")

In [ ]:
# EPR at the fault bus, per scenario
df = (
    study.compare_buses()
    .filter((pl.col("frequency_Hz") == "RMS") & (pl.col("bus_name") == "bus_mv_5"))
    .select(["scenario", "value"])
)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(df["scenario"].to_list(), df["value"].to_list())
ax.set_ylabel("EPR at fault bus (V, RMS)")
ax.set_title("Outage comparison at bus_mv_5")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In both outage scenarios the infeed at `bus_mv_1` no longer reaches the fault
at `bus_mv_5`, because the feeder is radial: no current flows to the fault,
and every earth potential rise is 0 V — the physically correct answer for an
islanded fault, not a failed solve.

## Summary

- `preview_pandapower_import` validates the mapping before any `Network` is
  built. Use it as the first step on real, larger pandapower networks.
- `from_pandapower` returns a `Network` with the imported buses and branches;
  `in_service=False` on the pandapower side becomes `active=False` on the
  groundinsight side, so outage scenarios start from the real grid state.
- The result plugs straight into `run_fault` and into `run_outage_study`.

**Next steps for a real import:**

- Replace the illustrative `ImportDefaults` with the cable and overhead-line
  types of the actual assets
  ([Bus and branch types from data sheets](../../howto/types-from-datasheets/)).
- Load an existing case with `pp.from_json` / `pp.from_pickle` before calling
  `preview_pandapower_import` and `from_pandapower`.
- Transformers are not modelled: import one voltage level at a time.
- A solved pandapower short-circuit case can be read as IEC 60909 quantities
  with `gi.read_shortcircuit_results` and applied with
  `gi.apply_shortcircuit_characteristics`
  ([I/O — network importers](../../api/io/#short-circuit-characteristics-iec-60909-0)).